# Задача 1. Compare

**Объяснение решения**

`merge_sort` делит массив пополам и рекурсивно сортирует обе части. Затем мерж: берем меньший из первых двух подмассивов.

`quick_sort` берет первый элемент как опорный. Делит массив на три части: меньше опорного, равные ему и больше. Первую и третью части сортирует рекурсивно, затем соединяет все три.

**Решение**

In [1]:
def merge_sort(nums):
    if len(nums) <= 1:
        return nums.copy()

    middle = len(nums) // 2
    left = merge_sort(nums[:middle])
    right = merge_sort(nums[middle:])

    result = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    result.extend(left[i:])
    result.extend(right[j:])
    return result

In [2]:
def quick_sort(nums):
    if len(nums) <= 1:
        return nums.copy()

    pivot = nums[0]
    left = [value for value in nums if value < pivot]
    equal = [value for value in nums if value == pivot]
    right = [value for value in nums if value > pivot]

    return quick_sort(left) + equal + quick_sort(right)

Декоратор возвращает пару `(результат, время в секундах)`. Для времени используем `perf_counter`.

Оборачиваем только внешний вызов сортировки. Внутри рекурсии остаются обычные функции, которые возвращают списки.

In [3]:
import time
from functools import wraps


def timer(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = function(*args, **kwargs)
        elapsed = time.perf_counter() - start
        return result, elapsed

    return wrapper


timed_merge_sort = timer(merge_sort)
timed_quick_sort = timer(quick_sort)

**Пример**

Возьмем `[4, -2, 7, 4, 0, 9, -5, 2, 7, 1]`. Здесь есть отрицательные числа, ноль и повторы.

In [4]:
nums = [4, -2, 7, 4, 0, 9, -5, 2, 7, 1]

print("Исходный массив:", nums)
print("Merge sort:", merge_sort(nums))
print("Quick sort:", quick_sort(nums))

Исходный массив: [4, -2, 7, 4, 0, 9, -5, 2, 7, 1]
Merge sort: [-5, -2, 0, 1, 2, 4, 4, 7, 7, 9]
Quick sort: [-5, -2, 0, 1, 2, 4, 4, 7, 7, 9]


**Merge sort**

Сначала делим массив до отдельных элементов:

```text
[4, -2, 7, 4, 0, 9, -5, 2, 7, 1]
├── [4, -2, 7, 4, 0]
│   ├── [4, -2] → [4] + [-2]
│   └── [7, 4, 0]
│       ├── [7]
│       └── [4, 0] → [4] + [0]
└── [9, -5, 2, 7, 1]
    ├── [9, -5] → [9] + [-5]
    └── [2, 7, 1]
        ├── [2]
        └── [7, 1] → [7] + [1]
```

На обратном пути сливаем части. Порядок строк совпадает с порядком выполнения:

| Шаг | Левая часть | Правая часть | Результат слияния |
|---|---|---|---|
| 1 | `[4]` | `[-2]` | `[-2, 4]` |
| 2 | `[4]` | `[0]` | `[0, 4]` |
| 3 | `[7]` | `[0, 4]` | `[0, 4, 7]` |
| 4 | `[-2, 4]` | `[0, 4, 7]` | `[-2, 0, 4, 4, 7]` |
| 5 | `[9]` | `[-5]` | `[-5, 9]` |
| 6 | `[7]` | `[1]` | `[1, 7]` |
| 7 | `[2]` | `[1, 7]` | `[1, 2, 7]` |
| 8 | `[-5, 9]` | `[1, 2, 7]` | `[-5, 1, 2, 7, 9]` |
| 9 | `[-2, 0, 4, 4, 7]` | `[-5, 1, 2, 7, 9]` | `[-5, -2, 0, 1, 2, 4, 4, 7, 7, 9]` |

**Quick sort**

Разбиения в порядке вызовов. Части длины `0` или `1` сразу возвращаются, равные опорному элементы больше не сортируем.

| Глубина | Массив | Опорный | Меньше | Равны | Больше |
|---|---|---|---|---|---|
| 0 | `[4, -2, 7, 4, 0, 9, -5, 2, 7, 1]` | `4` | `[-2, 0, -5, 2, 1]` | `[4, 4]` | `[7, 9, 7]` |
| 1 | `[-2, 0, -5, 2, 1]` | `-2` | `[-5]` | `[-2]` | `[0, 2, 1]` |
| 2 | `[0, 2, 1]` | `0` | `[]` | `[0]` | `[2, 1]` |
| 3 | `[2, 1]` | `2` | `[1]` | `[2]` | `[]` |
| 1 | `[7, 9, 7]` | `7` | `[]` | `[7, 7]` | `[9]` |

После рекурсивных вызовов соединяем части:

| Шаг | Соединение | Результат |
|---|---|---|
| 1 | `[1] + [2] + []` | `[1, 2]` |
| 2 | `[] + [0] + [1, 2]` | `[0, 1, 2]` |
| 3 | `[-5] + [-2] + [0, 1, 2]` | `[-5, -2, 0, 1, 2]` |
| 4 | `[] + [7, 7] + [9]` | `[7, 7, 9]` |
| 5 | `[-5, -2, 0, 1, 2] + [4, 4] + [7, 7, 9]` | `[-5, -2, 0, 1, 2, 4, 4, 7, 7, 9]` |

**Сложность**

Пусть `n` — длина массива, сравнение чисел занимает `O(1)`. Память указана для этих реализаций, включая новые списки и стек.

| Алгоритм и случай | Время | Дополнительная память |
|---|---|---|
| Merge sort, любой порядок | `O(n log n)` | `O(n)` |
| Quick sort, в среднем на случайных различных числах | `O(n log n)` | `O(n)` |
| Quick sort, упорядоченные различные числа | `O(n²)` | `O(n²)` |
| Quick sort, все элементы равны | `O(n)` | `O(n)` |
| Декоратор, без самой функции | `O(1)` | `O(1)` |

У merge sort `O(log n)` уровней, на каждом суммарно обрабатываем `O(n)` элементов. Одновременно храним списки суммарной длины `O(n)`.

У quick sort при равномерных разбиениях тоже `O(log n)` уровней. В худшем случае каждый раз остается часть длины `n - 1`: работа и сохраненные списки дают сумму `n + (n - 1) + ... + 1`, то есть `O(n²)`. Глубина стека тогда `O(n)`, а при равномерных разбиениях — `O(log n)`.

Если все числа равны, quick sort делает одно разбиение за `O(n)`. Пустой и одноэлементный массивы обе функции обрабатывают за `O(1)`.

**Тесты**

In [5]:
import random
import unittest
from itertools import product
from unittest.mock import patch


class TestSort(unittest.TestCase):
    def check(self, nums):
        before = nums.copy()
        expected = sorted(nums)

        for function in (merge_sort, quick_sort):
            with self.subTest(function=function.__name__):
                result = function(nums)
                self.assertEqual(result, expected)
                self.assertEqual(nums, before)
                self.assertIsNot(result, nums)

    def test_empty(self):
        self.check([])

    def test_single(self):
        self.check([7])

    def test_two_elements(self):
        for nums in ([1, 2], [2, 1], [2, 2]):
            self.check(nums)

    def test_example(self):
        self.check([4, -2, 7, 4, 0, 9, -5, 2, 7, 1])

    def test_duplicates(self):
        self.check([3, 1, 3, -2, 1, 3, -2, 0])
        self.check([5] * 5000)

    def test_sorted(self):
        self.check(list(range(600)))

    def test_reversed(self):
        self.check(list(range(599, -1, -1)))

    def test_negative_and_zero(self):
        self.check([0, -9, -1, 0, -5, -9])

    def test_large_numbers(self):
        self.check([10**100, 0, -10**100, 7, -7])

    def test_floats(self):
        self.check([1.5, -0.5, 2.0, 0.0, 1.5])

    def test_random(self):
        rng = random.Random(42)
        for size in (3, 10, 100, 1000, 5000):
            self.check([rng.randint(-1000, 1000) for _ in range(size)])

    def test_all_small_arrays(self):
        for size in range(7):
            for values in product((-1, 0, 1), repeat=size):
                self.check(list(values))

    def test_independent_results(self):
        for function in (merge_sort, quick_sort):
            nums = [3, 1, 2]
            first = function(nums)
            first[0] = 99
            self.assertEqual(nums, [3, 1, 2])
            self.assertEqual(function(nums), [1, 2, 3])


class TestTimer(unittest.TestCase):
    def test_arguments_and_time(self):
        calls = []

        @timer
        def add(a, b=0):
            calls.append((a, b))
            return a + b

        with patch("time.perf_counter", side_effect=[10.0, 10.125]):
            result, elapsed = add(3, b=5)

        self.assertEqual(result, 8)
        self.assertEqual(elapsed, 0.125)
        self.assertEqual(calls, [(3, 5)])
        self.assertEqual(add.__name__, "add")

    def test_same_result_object(self):
        values = [1, 2, 3]

        @timer
        def get_values():
            return values

        result, elapsed = get_values()
        self.assertIs(result, values)
        self.assertGreaterEqual(elapsed, 0)

    def test_exception(self):
        @timer
        def fail():
            raise ValueError("ошибка")

        with self.assertRaisesRegex(ValueError, "ошибка"):
            fail()

    def test_timed_sorts(self):
        nums = [4, -2, 7, 4, 0]
        for function in (timed_merge_sort, timed_quick_sort):
            result, elapsed = function(nums)
            self.assertEqual(result, [-2, 0, 4, 4, 7])
            self.assertGreaterEqual(elapsed, 0)
        self.assertEqual(nums, [4, -2, 7, 4, 0])

In [6]:
suite = unittest.TestSuite([
    unittest.defaultTestLoader.loadTestsFromTestCase(TestSort),
    unittest.defaultTestLoader.loadTestsFromTestCase(TestTimer),
])
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful(), "Некоторые тесты завершились ошибкой"
print(f"Успешно пройдено тестов: {test_result.testsRun}")

test_all_small_arrays (__main__.TestSort.test_all_small_arrays) ... ok
test_duplicates (__main__.TestSort.test_duplicates) ... ok
test_empty (__main__.TestSort.test_empty) ... ok
test_example (__main__.TestSort.test_example) ... ok
test_floats (__main__.TestSort.test_floats) ... ok
test_independent_results (__main__.TestSort.test_independent_results) ... ok
test_large_numbers (__main__.TestSort.test_large_numbers) ... ok
test_negative_and_zero (__main__.TestSort.test_negative_and_zero) ... ok
test_random (__main__.TestSort.test_random) ... ok
test_reversed (__main__.TestSort.test_reversed) ... ok
test_single (__main__.TestSort.test_single) ... ok
test_sorted (__main__.TestSort.test_sorted) ... ok
test_two_elements (__main__.TestSort.test_two_elements) ... ok
test_arguments_and_time (__main__.TestTimer.test_arguments_and_time) ... ok
test_exception (__main__.TestTimer.test_exception) ... ok
test_same_result_object (__main__.TestTimer.test_same_result_object) ... ok
test_timed_sorts (__m

Успешно пройдено тестов: 17


**Сравнение времени**

Берем одинаковые массивы для обеих сортировок. Делаем один прогрев и `9` замеров, сравниваем медианы. Порядок запуска сортировок чередуем. Создание массивов, проверка результата и вывод таблицы в замер не входят.

Проверим случайный порядок, возрастание, убывание, одинаковые числа и смесь пяти значений. Размеры — `200`, `400` и `600`. На упорядоченных данных глубина quick sort равна `n`, поэтому берем размеры ниже стандартного лимита рекурсии Python.

В последнем столбце делим время quick sort на время merge sort: больше `1` — quick sort медленнее, меньше `1` — быстрее. Конкретные времена зависят от компьютера.

In [7]:
from statistics import median
from IPython.display import Markdown, display


def benchmark(nums, repeats=9):
    functions = (timed_merge_sort, timed_quick_sort)
    expected = sorted(nums)
    times = [[], []]

    for function in functions:
        assert function(nums)[0] == expected

    for repeat in range(repeats):
        order = (0, 1) if repeat % 2 == 0 else (1, 0)
        for index in order:
            result, elapsed = functions[index](nums)
            assert result == expected
            times[index].append(elapsed)

    return median(times[0]), median(times[1])


measurements = []

for size in (200, 400, 600):
    rng = random.Random(42)
    cases = {
        "Случайный порядок": rng.sample(range(size), size),
        "По возрастанию": list(range(size)),
        "По убыванию": list(range(size - 1, -1, -1)),
        "Все равны": [7] * size,
        "Пять значений": [rng.randrange(5) for _ in range(size)],
    }

    for name, nums in cases.items():
        merge_time, quick_time = benchmark(nums)
        measurements.append((size, name, merge_time, quick_time))

rows = [
    "| n | Массив | Merge sort, мс | Quick sort, мс | Quick / Merge |",
    "|---|---|---|---|---|",
]

for size, name, merge_time, quick_time in measurements:
    rows.append(
        f"| {size} | {name} | {merge_time * 1000:.4f} | "
        f"{quick_time * 1000:.4f} | {quick_time / merge_time:.2f} |"
    )

display(Markdown("\n".join(rows)))

| n | Массив | Merge sort, мс | Quick sort, мс | Quick / Merge |
|---|---|---|---|---|
| 200 | Случайный порядок | 0.1132 | 0.0857 | 0.76 |
| 200 | По возрастанию | 0.0907 | 0.5883 | 6.49 |
| 200 | По убыванию | 0.0930 | 0.5738 | 6.17 |
| 200 | Все равны | 0.0891 | 0.0052 | 0.06 |
| 200 | Пять значений | 0.1133 | 0.0163 | 0.14 |
| 400 | Случайный порядок | 0.2611 | 0.1708 | 0.65 |
| 400 | По возрастанию | 0.1919 | 2.2368 | 11.66 |
| 400 | По убыванию | 0.1983 | 2.2596 | 11.40 |
| 400 | Все равны | 0.1775 | 0.0087 | 0.05 |
| 400 | Пять значений | 0.2512 | 0.0273 | 0.11 |
| 600 | Случайный порядок | 0.4487 | 0.2963 | 0.66 |
| 600 | По возрастанию | 0.2941 | 4.8296 | 16.42 |
| 600 | По убыванию | 0.3196 | 4.8826 | 15.28 |
| 600 | Все равны | 0.2870 | 0.0126 | 0.04 |
| 600 | Пять значений | 0.4044 | 0.0472 | 0.12 |

На упорядоченных различных числах quick sort каждый раз выбирает крайнее по величине значение и делит массив на части длины `0` и `n - 1`. Получаем `O(n²)`, а у merge sort остается `O(n log n)`.

На одинаковых числах quick sort собирает все элементы в `equal` за одно разбиение. Merge sort продолжает деление и слияние. Поэтому здесь преимущество у quick sort.

Этот результат зависит от выбора первого элемента опорным и разбиения на три части.